In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 92.3 MB/s eta 0:00:00


In [3]:
!pip install google-generativeai
!pip install yfinance

INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of proto-plus to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.3 MB/s

In [6]:
!pip install langchain-text-splitters

In [7]:
import pandas as pd
import numpy as np
from google import genai
import faiss
import yfinance as yf
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
ticker = "BMW.SW"
stock = yf.Ticker(ticker)

In [10]:
historical_data=stock.history(period="6mo")
print(historical_data.tail())
print(historical_data.head(2))

                                Open       High        Low      Close  Volume  \
Date                                                                            
2026-10-05 00:00:00+02:00  51.840000  51.840000  51.840000  51.840000       0   
2026-10-06 00:00:00+02:00  51.060001  51.060001  50.439999  50.500000      41   
2026-10-07 00:00:00+02:00  50.880001  50.880001  50.880001  50.880001       8   
2026-10-08 00:00:00+02:00  49.599998  49.599998  48.540001  48.540001     125   
2026-10-09 00:00:00+02:00  49.330002  49.330002  49.330002  49.330002       9   

                           Dividends  Stock Splits  
Date                                                
2026-10-05 00:00:00+02:00        0.0           0.0  
2026-10-06 00:00:00+02:00        0.0           0.0  
2026-10-07 00:00:00+02:00        0.0           0.0  
2026-10-08 00:00:00+02:00        0.0           0.0  
2026-10-09 00:00:00+02:00        0.0           0.0  
                                Open       High        Low   

In [11]:
company_info=stock.info
print(company_info)

{'address1': 'Petuelring 130', 'city': 'Munich', 'zip': '80809', 'country': 'Germany', 'phone': '49 89 382 0', 'fax': '49 89 382 70 25858', 'website': 'https://www.bmwgroup.com', 'industry': 'Auto Manufacturers', 'industryKey': 'auto-manufacturers', 'industryDisp': 'Auto Manufacturers', 'sector': 'Consumer Cyclical', 'sectorKey': 'consumer-cyclical', 'sectorDisp': 'Consumer Cyclical', 'longBusinessSummary': 'Bayerische Motoren Werke Aktiengesellschaft develops, manufactures, and sells automobiles and motorcycles, spare parts, and accessories worldwide. It operates through Automotive, Motorcycles, and Financial Services segments. The company offers automobiles under the BMW, MINI, and Rolls-Royce brands. It also develops, manufactures, and sells scooters, and motorcycles for private use and special-purpose vehicles for operational use under the BMW Motorrad brand. In addition, the company provides car rentals; credit financing; leasing that include insurance and service products; financ

In [14]:
news = stock.news
print(news)

[]


In [16]:
#create a financial knowledge base
#combine historical data + company info
documents=[]
company_text= f"""
Industry : {company_info['industry']}
Sector : {company_info['sector']}
Description:{company_info['longBusinessSummary']}
"""
documents.append(company_text)
print(documents)

['\nIndustry : Auto Manufacturers\nSector : Consumer Cyclical\nDescription:Bayerische Motoren Werke Aktiengesellschaft develops, manufactures, and sells automobiles and motorcycles, spare parts, and accessories worldwide. It operates through Automotive, Motorcycles, and Financial Services segments. The company offers automobiles under the BMW, MINI, and Rolls-Royce brands. It also develops, manufactures, and sells scooters, and motorcycles for private use and special-purpose vehicles for operational use under the BMW Motorrad brand. In addition, the company provides car rentals; credit financing; leasing that include insurance and service products; financing for dealership and customer deposits; vehicle fleet financing services for corporate car fleets under the Alphabet brand; and financial services for the automotive sector. It sells its products through retail outlet, agent, and independent dealerships. Bayerische Motoren Werke Aktiengesellschaft was founded in 1916 and is headquart

In [18]:
#historicaldata
historical_data_text=historical_data.tail(30).to_string()
documents.append(historical_data_text)
print(documents)

['\nIndustry : Auto Manufacturers\nSector : Consumer Cyclical\nDescription:Bayerische Motoren Werke Aktiengesellschaft develops, manufactures, and sells automobiles and motorcycles, spare parts, and accessories worldwide. It operates through Automotive, Motorcycles, and Financial Services segments. The company offers automobiles under the BMW, MINI, and Rolls-Royce brands. It also develops, manufactures, and sells scooters, and motorcycles for private use and special-purpose vehicles for operational use under the BMW Motorrad brand. In addition, the company provides car rentals; credit financing; leasing that include insurance and service products; financing for dealership and customer deposits; vehicle fleet financing services for corporate car fleets under the Alphabet brand; and financial services for the automotive sector. It sells its products through retail outlet, agent, and independent dealerships. Bayerische Motoren Werke Aktiengesellschaft was founded in 1916 and is headquart

In [19]:
print(documents[0])


Industry : Auto Manufacturers
Sector : Consumer Cyclical
Description:Bayerische Motoren Werke Aktiengesellschaft develops, manufactures, and sells automobiles and motorcycles, spare parts, and accessories worldwide. It operates through Automotive, Motorcycles, and Financial Services segments. The company offers automobiles under the BMW, MINI, and Rolls-Royce brands. It also develops, manufactures, and sells scooters, and motorcycles for private use and special-purpose vehicles for operational use under the BMW Motorrad brand. In addition, the company provides car rentals; credit financing; leasing that include insurance and service products; financing for dealership and customer deposits; vehicle fleet financing services for corporate car fleets under the Alphabet brand; and financial services for the automotive sector. It sells its products through retail outlet, agent, and independent dealerships. Bayerische Motoren Werke Aktiengesellschaft was founded in 1916 and is headquartered 

In [20]:
#textsplitting
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks = text_splitter.create_documents(documents)
print(chunks)

[Document(metadata={}, page_content='Industry : Auto Manufacturers\nSector : Consumer Cyclical'), Document(metadata={}, page_content='Description:Bayerische Motoren Werke Aktiengesellschaft develops, manufactures, and sells automobiles and motorcycles, spare parts, and accessories worldwide. It operates through Automotive, Motorcycles, and Financial Services segments. The company offers automobiles under the BMW, MINI, and Rolls-Royce brands. It also develops, manufactures, and sells scooters, and motorcycles for private use and special-purpose vehicles for operational use under the BMW Motorrad brand. In addition, the company provides car rentals; credit financing; leasing that include insurance and service products; financing for dealership and customer deposits; vehicle fleet financing services for corporate car fleets under the Alphabet brand; and financial services for the automotive sector. It sells its products through retail outlet, agent, and independent dealerships. Bayerisch

In [21]:
print(len(chunks))

6


In [23]:
#embeddings of the chunks
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
text=[doc.page_content for doc in chunks]
embeddings=embedding_model.encode(text)
print(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[ 0.00797833 -0.0511442  -0.00359953 ... -0.03867835  0.0796488
   0.03637543]
 [-0.03717196  0.03838798 -0.02414789 ... -0.08321641  0.03048502
   0.0469801 ]
 [ 0.01783804 -0.08208159 -0.01443953 ... -0.0769474  -0.09566379
   0.0556481 ]
 [ 0.0460227  -0.06227662 -0.02116315 ... -0.02651204 -0.1258136
  -0.01301593]
 [ 0.06989836 -0.08491226 -0.02408853 ... -0.03503774 -0.12767763
  -0.00477871]
 [ 0.06455366 -0.05802289 -0.02116264 ... -0.01261864 -0.12515453
  -0.01592285]]
